# Bibliotecas

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.colors as pc
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeClassifierCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans, BisectingKMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, normalized_mutual_info_score, make_scorer, f1_score, precision_score, recall_score
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_validate
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC

from scipy.signal import find_peaks
from scipy.integrate import trapezoid
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import skew, kurtosis, gaussian_kde, mannwhitneyu, ks_2samp, pearsonr, spearmanr, kendalltau, kruskal
from scipy import stats

from hampel import hampel

from scikit_posthocs import posthoc_dunn

from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

DIR_DATA = os.getcwd()+"/data/"
DIR_OUTPUT = os.getcwd()+"/output/"

# Carregando dados

## Crystallizer #1

In [ ]:
base_name_crystallizer1 = "Crystallizer #1.csv"

df_crystallizer1 = pd.read_csv(DIR_DATA + base_name_crystallizer1, sep=";", decimal=".")
df_crystallizer1["TIMESTAMP"] = pd.to_datetime(df_crystallizer1["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer1["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer1["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer1.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer1 = df_crystallizer1[df_crystallizer1.duplicated(subset=['Labref'], keep=False)]

# Retirando linhas 23 e 2141 que estão duplicadas mas não possuem amostras significativas
df_crystallizer1.drop([23,2141], inplace=True) 
# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer1[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer1.columns if col != 'Labref'}
df_crystallizer1 = df_crystallizer1.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer1 = df_crystallizer1.drop(remove.index)
df_crystallizer1

### Removendo outliers 0 a 10 #1

In [ ]:
df_crystallizer1_0a10 = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"].between(0, 10)].reset_index(drop=True)

print(f"Amostras originais : {len(df_crystallizer1)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer1_0a10)} ({len(df_crystallizer1) - len(df_crystallizer1_0a10)} removidas)")

df_crystallizer1_0a10

### Removendo outliers com IQR #1

In [ ]:
serie = df_crystallizer1["Resultado de Ferro (ppm)"]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

df_crystallizer1_iqr = df_crystallizer1[serie.between(limite_inf, limite_sup)].reset_index(drop=True)

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer1)}")
print(f"Amostras removidas: {len(df_crystallizer1) - len(df_crystallizer1_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer1_iqr)}")

df_crystallizer1_iqr

### Removendo outliers com Filtro de Hampel #1

In [ ]:
serie = df_crystallizer1["Resultado de Ferro (ppm)"].reset_index(drop=True)

resultado = hampel(serie, window_size=7, n_sigma=3.0)

serie_filtrada = resultado.filtered_data
outlier_indices = resultado.outlier_indices

df_crystallizer1_hampel = df_crystallizer1.copy().reset_index(drop=True)
df_crystallizer1_hampel["Resultado de Ferro (ppm)"] = serie_filtrada

print(f"Outliers detectados: {len(outlier_indices)}")
df_crystallizer1_hampel

## Crystallizer #2

In [ ]:
base_name_crystallizer2 = "Crystallizer #2.csv"

df_crystallizer2 = pd.read_csv(DIR_DATA + base_name_crystallizer2, sep=";", decimal=".")
df_crystallizer2["TIMESTAMP"] = pd.to_datetime(df_crystallizer2["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer2["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer2["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer2.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer2 = df_crystallizer2[df_crystallizer2.duplicated(subset=['Labref'], keep=False)]

# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer2[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer2.columns if col != 'Labref'}
df_crystallizer2 = df_crystallizer2.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostra 4027521 que possui valor muito discrepante
idx = df_crystallizer2[df_crystallizer2['Labref'] == 4027521].index 
df_crystallizer2 = df_crystallizer2.drop(idx)


# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer2 = df_crystallizer2.drop(remove.index)
df_crystallizer2

### Removendo outliers 0 a 10 #2

In [ ]:
df_crystallizer2_0a10 = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"].between(0, 10)].reset_index(drop=True)

print(f"Amostras originais : {len(df_crystallizer2)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer2_0a10)} ({len(df_crystallizer2) - len(df_crystallizer2_0a10)} removidas)")

df_crystallizer2_0a10

### Removendo outliers com IQR #2

In [ ]:
serie = df_crystallizer2["Resultado de Ferro (ppm)"]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

df_crystallizer2_iqr = df_crystallizer2[serie.between(limite_inf, limite_sup)].reset_index(drop=True)

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer2)}")
print(f"Amostras removidas: {len(df_crystallizer2) - len(df_crystallizer2_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer2_iqr)}")

df_crystallizer2_iqr

### Removendo outliers com Filtro de Hampel #2

In [ ]:
serie = df_crystallizer2["Resultado de Ferro (ppm)"].reset_index(drop=True)

resultado = hampel(serie, window_size=7, n_sigma=3.0)

serie_filtrada = resultado.filtered_data
outlier_indices = resultado.outlier_indices

df_crystallizer2_hampel = df_crystallizer2.copy().reset_index(drop=True)
df_crystallizer2_hampel["Resultado de Ferro (ppm)"] = serie_filtrada

print(f"Outliers detectados: {len(outlier_indices)}")
df_crystallizer2_hampel

## Crystallizer #3

In [ ]:
base_name_crystallizer3 = "Crystallizer #3.csv"

df_crystallizer3 = pd.read_csv(DIR_DATA + base_name_crystallizer3, sep=";", decimal=".")
df_crystallizer3["TIMESTAMP"] = pd.to_datetime(df_crystallizer3["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer3["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer3["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer3.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer3 = df_crystallizer3[df_crystallizer3.duplicated(subset=['Labref'], keep=False)]


# Retirando linhas 6476 e 6494 que estão duplicadas mas não possuem amostras significativas
df_crystallizer3.drop([6476,6494], inplace=True) 
# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer3[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer3.columns if col != 'Labref'}
df_crystallizer3 = df_crystallizer3.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer3 = df_crystallizer3.drop(remove.index)
df_crystallizer3

### Removendo outliers 0 a 10 #3

In [ ]:
df_crystallizer3_0a10 = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"].between(0, 10)].reset_index(drop=True)

print(f"Amostras originais : {len(df_crystallizer3)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer3_0a10)} ({len(df_crystallizer3) - len(df_crystallizer3_0a10)} removidas)")

df_crystallizer3_0a10

### Removendo outliers com IQR #3

In [ ]:
serie = df_crystallizer3["Resultado de Ferro (ppm)"]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

df_crystallizer3_iqr = df_crystallizer3[serie.between(limite_inf, limite_sup)].reset_index(drop=True)

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer3)}")
print(f"Amostras removidas: {len(df_crystallizer3) - len(df_crystallizer3_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer3_iqr)}")

df_crystallizer3_iqr

### Removendo outliers com Filtro de Hampel #3

In [ ]:
serie = df_crystallizer3["Resultado de Ferro (ppm)"].reset_index(drop=True)

resultado = hampel(serie, window_size=7, n_sigma=3.0)

serie_filtrada = resultado.filtered_data
outlier_indices = resultado.outlier_indices

df_crystallizer3_hampel = df_crystallizer3.copy().reset_index(drop=True)
df_crystallizer3_hampel["Resultado de Ferro (ppm)"] = serie_filtrada

print(f"Outliers detectados: {len(outlier_indices)}")
df_crystallizer3_hampel

# Carregando eventos identificados

## Deifinindo LC

In [ ]:
threshold = 5

## Crystallizer #1

In [ ]:
base_name_eventos_crystallizer1 = "Eventos-Reator1.csv"

df_eventos_crystallizer1 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer1, sep=";", decimal=".")
df_eventos_crystallizer1["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer1["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer1["Real"] = 1

nova_linha = {
    "TIMESTAMP": pd.to_datetime("2025-10-21 08:38:00"),
    "Real": 0,
    "Evento": "Falso Alarme"
}

df_eventos_crystallizer1 = pd.concat([df_eventos_crystallizer1, pd.DataFrame([nova_linha])], ignore_index=True)
df_eventos_crystallizer1 = df_eventos_crystallizer1.sort_values("TIMESTAMP").reset_index(drop=True)

# Removendo eventos que não são trocas de reator
df_eventos_crystallizer1.drop([2,3,4,6,7,8], inplace=True) 
df_eventos_crystallizer1

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer1 original
    diffs = (df_eventos_crystallizer1["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer1 existente
df_eventos_crystallizer1 = pd.concat([df_eventos_crystallizer1, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer1 = df_eventos_crystallizer1.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer1

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer1_filtrado_until2020 = df_eventos_crystallizer1[df_eventos_crystallizer1["TIMESTAMP"] <= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer1_filtrado_until2020.head()

In [ ]:
df_eventos_crystallizer1_filtrado_after2020 = df_eventos_crystallizer1[df_eventos_crystallizer1["TIMESTAMP"] >= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer1_filtrado_after2020.head()

## Crystallizer #2

In [ ]:
base_name_eventos_crystallizer2 = "Eventos-Reator2.csv"

df_eventos_crystallizer2 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer2, sep=";", decimal=".")
df_eventos_crystallizer2["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer2["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer2["Real"] = 1

# Removendo eventos fora do período de dados
df_eventos_crystallizer2.drop([0,1,2], inplace=True) 

# Removendo eventos que não são trocas de reator
df_eventos_crystallizer2.drop([6,9], inplace=True) 
df_eventos_crystallizer2

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer2 original
    diffs = (df_eventos_crystallizer2["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer2 existente
df_eventos_crystallizer2 = pd.concat([df_eventos_crystallizer2, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer2 = df_eventos_crystallizer2.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer2

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer2_filtrado_until2020 = df_eventos_crystallizer2[df_eventos_crystallizer2["TIMESTAMP"] <= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer2_filtrado_until2020.head()

In [ ]:
df_eventos_crystallizer2_filtrado_after2020 = df_eventos_crystallizer2[df_eventos_crystallizer2["TIMESTAMP"] >= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer2_filtrado_after2020.head()

## Crystallizer #3

In [ ]:
base_name_eventos_crystallizer3 = "Eventos-Reator3.csv"

df_eventos_crystallizer3 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer3, sep=";", decimal=".")
df_eventos_crystallizer3["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer3["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer3["Real"] = 1

df_eventos_crystallizer3.drop([0,1], inplace=True) # Removendo eventos fora do período de dados

# Removendo eventos que não são trocas de reator
df_eventos_crystallizer3.drop([4,7], inplace=True) 
df_eventos_crystallizer3

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer3 original
    diffs = (df_eventos_crystallizer3["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer3 existente
df_eventos_crystallizer3 = pd.concat([df_eventos_crystallizer3, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer3 = df_eventos_crystallizer3.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer3

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer3_filtrado_until2020 = df_eventos_crystallizer3[df_eventos_crystallizer3["TIMESTAMP"] <= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer3_filtrado_until2020.head()

In [ ]:
df_eventos_crystallizer3_filtrado_after2020 = df_eventos_crystallizer3[df_eventos_crystallizer3["TIMESTAMP"] >= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer3_filtrado_after2020.head()

# Plotando Gráfico das medições
Linhas vermelhas = Eventos relatados  
Linhas azuis = Eventos de ultapassagem sem relatos

## Crystallizer #1

In [ ]:
fig_crystallizer1 = go.Figure()
fig_crystallizer1.add_trace(go.Scatter(
    x=df_crystallizer1['TIMESTAMP'],
    y=df_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #1",
)
fig_crystallizer1.show()

### Crystallizer #1 - Outliers 0 a 10

In [ ]:
fig_crystallizer1_0a10 = go.Figure()
fig_crystallizer1_0a10.add_trace(go.Scatter(
    x=df_crystallizer1_0a10['TIMESTAMP'],
    y=df_crystallizer1_0a10["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer1_0a10.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer1_0a10.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer1_0a10.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer1_0a10.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #1 - Outliers 0 a 10",
)
fig_crystallizer1_0a10.show()

### Crystallizer #1 - Outliers IQR

In [ ]:
fig_crystallizer1_iqr = go.Figure()
fig_crystallizer1_iqr.add_trace(go.Scatter(
    x=df_crystallizer1_iqr['TIMESTAMP'],
    y=df_crystallizer1_iqr["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer1_iqr.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer1_iqr.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer1_iqr.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer1_iqr.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #1 - IQR",
)
fig_crystallizer1_iqr.show()

### Crystallizer #1 - Filtro de Hampel

In [ ]:
fig_crystallizer1_hampel = go.Figure()
fig_crystallizer1_hampel.add_trace(go.Scatter(
    x=df_crystallizer1_hampel['TIMESTAMP'],
    y=df_crystallizer1_hampel["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer1_hampel.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer1_hampel.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer1_hampel.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer1_hampel.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #1 - Filtro de Hampel",
)
fig_crystallizer1_hampel.show()

## Crystallizer #2

In [ ]:
fig_crystallizer2 = go.Figure()
fig_crystallizer2.add_trace(go.Scatter(
    x=df_crystallizer2['TIMESTAMP'],
    y=df_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #2",
)
fig_crystallizer2.show()

### Crystallizer #2 - Outliers 0 a 10

In [ ]:
fig_crystallizer2_0a10 = go.Figure()
fig_crystallizer2_0a10.add_trace(go.Scatter(
    x=df_crystallizer2_0a10['TIMESTAMP'],
    y=df_crystallizer2_0a10["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer2_0a10.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer2_0a10.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer2_0a10.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer2_0a10.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #2 - Outliers 0 a 10",
)
fig_crystallizer2_0a10.show()

### Crystallizer #2 - Outliers IQR

In [ ]:
fig_crystallizer2_iqr = go.Figure()
fig_crystallizer2_iqr.add_trace(go.Scatter(
    x=df_crystallizer2_iqr['TIMESTAMP'],
    y=df_crystallizer2_iqr["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer2_iqr.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer2_iqr.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer2_iqr.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer2_iqr.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #2 - IQR",
)
fig_crystallizer2_iqr.show()

### Crystallizer #2 - Filtro de Hampel

In [ ]:
fig_crystallizer2_hampel = go.Figure()
fig_crystallizer2_hampel.add_trace(go.Scatter(
    x=df_crystallizer2_hampel['TIMESTAMP'],
    y=df_crystallizer2_hampel["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer2_hampel.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer2_hampel.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer2_hampel.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer2_hampel.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #2 - Filtro de Hampel",
)
fig_crystallizer2_hampel.show()

## Crystallizer #3

In [ ]:
fig_crystallizer3 = go.Figure()
fig_crystallizer3.add_trace(go.Scatter(
    x=df_crystallizer3['TIMESTAMP'],
    y=df_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #3",
)
fig_crystallizer3.show()

### Crystallizer #3 - Outliers 0 a 10

In [ ]:
fig_crystallizer3_0a10 = go.Figure()
fig_crystallizer3_0a10.add_trace(go.Scatter(
    x=df_crystallizer3_0a10['TIMESTAMP'],
    y=df_crystallizer3_0a10["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer3_0a10.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer3_0a10.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer3_0a10.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer3_0a10.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #3 - Outliers 0 a 10",
)
fig_crystallizer3_0a10.show()

### Crystallizer #3 - Outliers IQR

In [ ]:
fig_crystallizer3_iqr = go.Figure()
fig_crystallizer3_iqr.add_trace(go.Scatter(
    x=df_crystallizer3_iqr['TIMESTAMP'],
    y=df_crystallizer3_iqr["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer3_iqr.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer3_iqr.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer3_iqr.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer3_iqr.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #3 - IQR",
)
fig_crystallizer3_iqr.show()

### Crystallizer #2 - Filtro de Hampel

In [ ]:
fig_crystallizer3_hampel = go.Figure()
fig_crystallizer3_hampel.add_trace(go.Scatter(
    x=df_crystallizer3_hampel['TIMESTAMP'],
    y=df_crystallizer3_hampel["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer3_hampel.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer3_hampel.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer3_hampel.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer3_hampel.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #3 - Filtro de Hampel",
)
fig_crystallizer3_hampel.show()

## Crystallizer #1#2#3

In [ ]:
crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = go.Figure()

for c in crystallizers:
    # Série principal
    fig.add_trace(go.Scatter(
        x=c["df"]['TIMESTAMP'],
        y=c["df"]["Resultado de Ferro (ppm)"],
        mode='lines',
        name=c["nome"],
        line=dict(color=c["cor"])
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        cor = c["cor"] if row["Real"] == 1 else "gray"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizers #1, #2 e #3"
)
fig.show()

### Outliers 0 a 10

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_0a10, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_0a10, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_0a10, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = go.Figure()

for c in crystallizers:
    # Série principal
    fig.add_trace(go.Scatter(
        x=c["df"]['TIMESTAMP'],
        y=c["df"]["Resultado de Ferro (ppm)"],
        mode='lines',
        name=c["nome"],
        line=dict(color=c["cor"])
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        cor = c["cor"] if row["Real"] == 1 else "gray"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizers #1, #2 e #3 - Outliers 0 a 10"
)
fig.show()

### Outliers IQR

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_iqr, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_iqr, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_iqr, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = go.Figure()

for c in crystallizers:
    # Série principal
    fig.add_trace(go.Scatter(
        x=c["df"]['TIMESTAMP'],
        y=c["df"]["Resultado de Ferro (ppm)"],
        mode='lines',
        name=c["nome"],
        line=dict(color=c["cor"])
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        cor = c["cor"] if row["Real"] == 1 else "gray"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizers #1, #2 e #3 - Outliers IQR"
)
fig.show()

### Outliers Hampel

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_hampel, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_hampel, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_hampel, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = go.Figure()

for c in crystallizers:
    # Série principal
    fig.add_trace(go.Scatter(
        x=c["df"]['TIMESTAMP'],
        y=c["df"]["Resultado de Ferro (ppm)"],
        mode='lines',
        name=c["nome"],
        line=dict(color=c["cor"])
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        cor = c["cor"] if row["Real"] == 1 else "gray"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizers #1, #2 e #3 - Filtro de Hampel"
)
fig.show()

## Gráfico com Média Móvel
Verificando se há tendência clara nos dados

## MM Crystallizer #1

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer1 = df_crystallizer1.copy()
df_mm_crystallizer1 = df_mm_crystallizer1.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer1[f'MM_{dias}D'] = df_mm_crystallizer1["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer1 = df_mm_crystallizer1.reset_index()

fig_mm_crystallizer1 = go.Figure()

# Série original
fig_mm_crystallizer1.add_trace(go.Scatter(
    x=df_mm_crystallizer1['TIMESTAMP'],
    y=df_mm_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer1.add_trace(go.Scatter(
        x=df_mm_crystallizer1['TIMESTAMP'],
        y=df_mm_crystallizer1[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #1"
)
fig_mm_crystallizer1.show()

### MM Crystallizer #1 - Outliers 0 a 10

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer1 = df_crystallizer1_0a10.copy()
df_mm_crystallizer1 = df_mm_crystallizer1.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer1[f'MM_{dias}D'] = df_mm_crystallizer1["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer1 = df_mm_crystallizer1.reset_index()

fig_mm_crystallizer1 = go.Figure()

# Série original
fig_mm_crystallizer1.add_trace(go.Scatter(
    x=df_mm_crystallizer1['TIMESTAMP'],
    y=df_mm_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer1.add_trace(go.Scatter(
        x=df_mm_crystallizer1['TIMESTAMP'],
        y=df_mm_crystallizer1[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #1 - Outliers 0 a 10"
)
fig_mm_crystallizer1.show()

### MM Crystallizer #1 - IQR

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer1 = df_crystallizer1_iqr.copy()
df_mm_crystallizer1 = df_mm_crystallizer1.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer1[f'MM_{dias}D'] = df_mm_crystallizer1["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer1 = df_mm_crystallizer1.reset_index()

fig_mm_crystallizer1 = go.Figure()

# Série original
fig_mm_crystallizer1.add_trace(go.Scatter(
    x=df_mm_crystallizer1['TIMESTAMP'],
    y=df_mm_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer1.add_trace(go.Scatter(
        x=df_mm_crystallizer1['TIMESTAMP'],
        y=df_mm_crystallizer1[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #1 - IQR"
)
fig_mm_crystallizer1.show()

### MM Crystallizer #1 - Filtro de Hampel

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer1 = df_crystallizer1_hampel.copy()
df_mm_crystallizer1 = df_mm_crystallizer1.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer1[f'MM_{dias}D'] = df_mm_crystallizer1["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer1 = df_mm_crystallizer1.reset_index()

fig_mm_crystallizer1 = go.Figure()

# Série original
fig_mm_crystallizer1.add_trace(go.Scatter(
    x=df_mm_crystallizer1['TIMESTAMP'],
    y=df_mm_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer1.add_trace(go.Scatter(
        x=df_mm_crystallizer1['TIMESTAMP'],
        y=df_mm_crystallizer1[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #1 - Filtro de Hampel"
)
fig_mm_crystallizer1.show()

## MM Crystallizer #2

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer2 = df_crystallizer2.copy()
df_mm_crystallizer2 = df_mm_crystallizer2.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer2[f'MM_{dias}D'] = df_mm_crystallizer2["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer2 = df_mm_crystallizer2.reset_index()

fig_mm_crystallizer2 = go.Figure()

# Série original
fig_mm_crystallizer2.add_trace(go.Scatter(
    x=df_mm_crystallizer2['TIMESTAMP'],
    y=df_mm_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer2.add_trace(go.Scatter(
        x=df_mm_crystallizer2['TIMESTAMP'],
        y=df_mm_crystallizer2[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #2"
)
fig_mm_crystallizer2.show()

### MM Crystallizer #2 - Outliers 0 a 10

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer2 = df_crystallizer2_0a10.copy()
df_mm_crystallizer2 = df_mm_crystallizer2.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer2[f'MM_{dias}D'] = df_mm_crystallizer2["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer2 = df_mm_crystallizer2.reset_index()

fig_mm_crystallizer2 = go.Figure()

# Série original
fig_mm_crystallizer2.add_trace(go.Scatter(
    x=df_mm_crystallizer2['TIMESTAMP'],
    y=df_mm_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer2.add_trace(go.Scatter(
        x=df_mm_crystallizer2['TIMESTAMP'],
        y=df_mm_crystallizer2[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #2 - Outiliers 0 a 10"
)
fig_mm_crystallizer2.show()

### MM Crystallizer #2 - IQR

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer2 = df_crystallizer2_iqr.copy()
df_mm_crystallizer2 = df_mm_crystallizer2.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer2[f'MM_{dias}D'] = df_mm_crystallizer2["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer2 = df_mm_crystallizer2.reset_index()

fig_mm_crystallizer2 = go.Figure()

# Série original
fig_mm_crystallizer2.add_trace(go.Scatter(
    x=df_mm_crystallizer2['TIMESTAMP'],
    y=df_mm_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer2.add_trace(go.Scatter(
        x=df_mm_crystallizer2['TIMESTAMP'],
        y=df_mm_crystallizer2[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #2 - IQR"
)
fig_mm_crystallizer2.show()

### MM Crystallizer #2 - Filtro de Hampel

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer2 = df_crystallizer2_hampel.copy()
df_mm_crystallizer2 = df_mm_crystallizer2.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer2[f'MM_{dias}D'] = df_mm_crystallizer2["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer2 = df_mm_crystallizer2.reset_index()

fig_mm_crystallizer2 = go.Figure()

# Série original
fig_mm_crystallizer2.add_trace(go.Scatter(
    x=df_mm_crystallizer2['TIMESTAMP'],
    y=df_mm_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer2.add_trace(go.Scatter(
        x=df_mm_crystallizer2['TIMESTAMP'],
        y=df_mm_crystallizer2[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #2 - Filtro de Hampel"
)
fig_mm_crystallizer2.show()

## MM Crystallizer #3

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer3 = df_crystallizer3.copy()
df_mm_crystallizer3 = df_mm_crystallizer3.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer3[f'MM_{dias}D'] = df_mm_crystallizer3["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer3 = df_mm_crystallizer3.reset_index()

fig_mm_crystallizer3 = go.Figure()

# Série original
fig_mm_crystallizer3.add_trace(go.Scatter(
    x=df_mm_crystallizer3['TIMESTAMP'],
    y=df_mm_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer3.add_trace(go.Scatter(
        x=df_mm_crystallizer3['TIMESTAMP'],
        y=df_mm_crystallizer3[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #3"
)
fig_mm_crystallizer3.show()

### MM Crystallizer #3 - Outliers 0 a 10

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer3 = df_crystallizer3_0a10.copy()
df_mm_crystallizer3 = df_mm_crystallizer3.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer3[f'MM_{dias}D'] = df_mm_crystallizer3["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer3 = df_mm_crystallizer3.reset_index()

fig_mm_crystallizer3 = go.Figure()

# Série original
fig_mm_crystallizer3.add_trace(go.Scatter(
    x=df_mm_crystallizer3['TIMESTAMP'],
    y=df_mm_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer3.add_trace(go.Scatter(
        x=df_mm_crystallizer3['TIMESTAMP'],
        y=df_mm_crystallizer3[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #3 - Outliers 0 a 10"
)
fig_mm_crystallizer3.show()

### MM Crystallizer #3 - IQR

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer3 = df_crystallizer3_iqr.copy()
df_mm_crystallizer3 = df_mm_crystallizer3.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer3[f'MM_{dias}D'] = df_mm_crystallizer3["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer3 = df_mm_crystallizer3.reset_index()

fig_mm_crystallizer3 = go.Figure()

# Série original
fig_mm_crystallizer3.add_trace(go.Scatter(
    x=df_mm_crystallizer3['TIMESTAMP'],
    y=df_mm_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer3.add_trace(go.Scatter(
        x=df_mm_crystallizer3['TIMESTAMP'],
        y=df_mm_crystallizer3[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #3 - IQR"
)
fig_mm_crystallizer3.show()

### MM Crystallizer #3 - Filtro de Hampel

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer3 = df_crystallizer3_hampel.copy()
df_mm_crystallizer3 = df_mm_crystallizer3.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer3[f'MM_{dias}D'] = df_mm_crystallizer3["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer3 = df_mm_crystallizer3.reset_index()

fig_mm_crystallizer3 = go.Figure()

# Série original
fig_mm_crystallizer3.add_trace(go.Scatter(
    x=df_mm_crystallizer3['TIMESTAMP'],
    y=df_mm_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer3.add_trace(go.Scatter(
        x=df_mm_crystallizer3['TIMESTAMP'],
        y=df_mm_crystallizer3[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #3 - Filtro de Hampel"
)
fig_mm_crystallizer3.show()

## MM Crystallizer #1#2#3

In [ ]:
NUM_DIAS = 7  # parâmetro da janela

CORES = {
    "Crystallizer #1": "blue",
    "Crystallizer #2": "green",
    "Crystallizer #3": "orange"
}

crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3"},
]

fig_mm_123 = go.Figure()

for c in crystallizers:
    cor = CORES[c["nome"]]

    # Calcula média móvel
    df_mm = c["df"].copy().set_index('TIMESTAMP')
    df_mm[f'MM_{NUM_DIAS}D'] = df_mm["Resultado de Ferro (ppm)"].rolling(window=f'{NUM_DIAS}D').mean()
    df_mm = df_mm.reset_index()

    # Série original
    fig_mm_123.add_trace(go.Scatter(
        x=df_mm['TIMESTAMP'],
        y=df_mm["Resultado de Ferro (ppm)"],
        mode='lines',
        name=f"{c['nome']} — original",
        line=dict(color=cor, width=1),
        opacity=0.3
    ))

    # Média móvel
    fig_mm_123.add_trace(go.Scatter(
        x=df_mm['TIMESTAMP'],
        y=df_mm[f'MM_{NUM_DIAS}D'],
        mode='lines',
        name=f"{c['nome']} — MM {NUM_DIAS}D",
        line=dict(color=cor, width=2)
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        dash_evento = "solid" if row["Real"] == 1 else "dash"
        fig_mm_123.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0, y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash=dash_evento)
        )
        fig_mm_123.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig_mm_123.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig_mm_123.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title=f"Fe (ppm) MM {NUM_DIAS}D — Crystallizers #1, #2 e #3"
)
fig_mm_123.show()

# Estatísticas descritivas de toda série de concentração de Fe

In [ ]:
def estatisticas(serie, nome):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    return pd.Series({
        "Contagem"     : serie.count(),
        "Média"        : serie.mean(),
        "Mediana"      : serie.median(),
        "Desvio Padrão": serie.std(),
        "Variância"    : serie.var(),
        "Mínimo"       : serie.min(),
        "Máximo"       : serie.max(),
        "Amplitude"    : serie.max() - serie.min(),
        "Q1 (25%)"     : Q1,
        "Q3 (75%)"     : Q3,
        "IQR"          : Q3 - Q1,
        "Assimetria"   : serie.skew(),
        "Curtose"      : serie.kurt()
    }, name=nome)

# Coluna separadora vazia
separador = pd.Series({k: "" for k in ["Contagem","Média","Mediana","Desvio Padrão","Variância",
                                        "Mínimo","Máximo","Amplitude","Q1 (25%)","Q3 (75%)","IQR",
                                        "Assimetria","Curtose"]})

c1 = pd.concat([
    estatisticas(df_crystallizer1["Resultado de Ferro (ppm)"].dropna(),       "C1 Original"),
    estatisticas(df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna(),  "C1 Intervalo 0-10"),
    estatisticas(df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna(),   "C1 IQR"),
    estatisticas(df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna(),"C1 Hampel"),
], axis=1)

c2 = pd.concat([
    estatisticas(df_crystallizer2["Resultado de Ferro (ppm)"].dropna(),       "C2 Original"),
    estatisticas(df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna(),  "C2 Intervalo 0-10"),
    estatisticas(df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna(),   "C2 IQR"),
    estatisticas(df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna(),"C2 Hampel"),
], axis=1)

c3 = pd.concat([
    estatisticas(df_crystallizer3["Resultado de Ferro (ppm)"].dropna(),       "C3 Original"),
    estatisticas(df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna(),  "C3 Intervalo 0-10"),
    estatisticas(df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna(),   "C3 IQR"),
    estatisticas(df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna(),"C3 Hampel"),
], axis=1)

sep = separador.rename("│")

df_comparativo = pd.concat([c1, sep, c2, sep.rename("│"), c3], axis=1).round(4)

# Corrige as colunas separadoras que ficaram com float após o round
df_comparativo["│"]  = ""
df_comparativo["│"] = ""

df_comparativo

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

CORES = {"C1": "#1f77b4", "C2": "#2ca02c", "C3": "#ff7f0e"}

def hex_to_rgba(cor, alpha=0.15):
    rgb = pc.hex_to_rgb(cor)
    return f'rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha})'

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=2,
    subplot_titles=[
        titulo
        for m in metodos
        for titulo in [f"KDE — {m['titulo']}", f"Violin Plot — {m['titulo']}"]
    ]
)

for row_idx, metodo in enumerate(metodos, start=1):
    for c in metodo["series"]:
        serie = c["serie"]
        nome  = c["nome"]
        cor   = CORES[nome]

        # KDE na escala de densidade natural (área sob a curva = 1)
        kde = gaussian_kde(serie)
        x_range = np.linspace(serie.min(), serie.max(), 1000)
        y_kde = kde(x_range)
        y_kde = y_kde / trapezoid(y_kde, x_range)  # normaliza

        fig.add_trace(go.Scatter(
            x=x_range,
            y=y_kde,
            mode='lines',
            line=dict(color=cor, width=2),
            fill='tozeroy',
            fillcolor=hex_to_rgba(cor, alpha=0.15),
            name=nome,
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

        # Violin
        fig.add_trace(go.Violin(
            y=serie,
            name=nome,
            marker_color=cor,
            fillcolor=hex_to_rgba(cor, alpha=0.4),
            box_visible=True,
            meanline_visible=True,
            legendgroup=nome,
            showlegend=False
        ), row=row_idx, col=2)

    fig.update_xaxes(title_text="Resultado de Ferro (ppm)", row=row_idx, col=1)
    fig.update_yaxes(title_text="Densidade", row=row_idx, col=1)
    fig.update_yaxes(title_text="ppm", row=row_idx, col=2)

fig.update_layout(
    height=500 * n_metodos,
    template='plotly_white',
    title="Análise Descritiva Comparativa — Resultado de Ferro (ppm)",
)
fig.show()

### Teste estatístico para verificar se distribuições são diferentes
Com n~30000 qualquer desvio nas amostras altera muito o p-value tornando o teste sensível a esses valores

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

resultados_testes = []

for metodo in metodos:
    series = [c["serie"].values for c in metodo["series"]]
    nomes  = [c["nome"] for c in metodo["series"]]

    # Kruskal-Wallis
    stat_kw, p_kw = kruskal(*series)

    # Dunn — formato longo com pd.concat de rows (funciona com séries de tamanhos diferentes)
    dados_dunn = pd.concat([
        pd.DataFrame({"grupo": nome, "valor": s})
        for nome, s in zip(nomes, series)
    ], ignore_index=True)

    dunn = posthoc_dunn(
        dados_dunn, val_col="valor", group_col="grupo",
        p_adjust="bonferroni"
    )

    resultados_testes.append({
        "Método"          : metodo["titulo"],
        "Kruskal-Wallis H": round(stat_kw, 4),
        "p-value"         : round(p_kw, 6),
        "Podem unificar?" : "✅ Sim" if p_kw > 0.05 else "❌ Não",
        "Dunn C1 vs C2"   : round(dunn.loc["C1", "C2"], 4),
        "Dunn C1 vs C3"   : round(dunn.loc["C1", "C3"], 4),
        "Dunn C2 vs C3"   : round(dunn.loc["C2", "C3"], 4),
    })

    print(f"\n{'='*55}")
    print(f"Método: {metodo['titulo']}")
    print(f"  Kruskal-Wallis H = {stat_kw:.4f}  |  p = {p_kw:.6f}")
    print(f"  {'✅ Distribuições compatíveis' if p_kw > 0.05 else '❌ Distribuições significativamente diferentes'}")
    print(f"\n  Dunn post-hoc (p-values corrigidos por Bonferroni):")
    print(dunn.round(4).to_string())

df_resultados_testes = pd.DataFrame(resultados_testes).set_index("Método")
print(f"\n{'='*55}")
df_resultados_testes

### Teste de Kruskal-Wallis com Effect Size (η²)

Para amostras grandes (n ~ 30.000), testes estatísticos como o Kruskal-Wallis tendem a rejeitar a hipótese nula mesmo com diferenças praticamente irrelevantes. Por isso o p-value é complementado pelo **eta-quadrado (η²)** que mede a proporção da variação total explicada pelo agrupamento por crystallizer.

| η²        | Interpretação                                      |
|-----------|----------------------------------------------------|
| < 0.01    | Efeito negligenciável — unificação justificada     |
| 0.01–0.06 | Efeito pequeno — unificação provavelmente aceitável|
| 0.06–0.14 | Efeito médio — avaliar com cautela                 |
| > 0.14    | Efeito grande — distribuições substancialmente diferentes |

A decisão de unificar os dados dos três crystallizers deve considerar em conjunto o η², o p-value e a inspeção visual das curvas KDE e violin plots gerados.

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

for metodo in metodos:
    series = [c["serie"].values for c in metodo["series"]]
    nomes  = [c["nome"] for c in metodo["series"]]
    n_total = sum(len(s) for s in series)

    stat_kw, p_kw = kruskal(*series)

    # Eta-quadrado: mede o quanto da variação total é explicada pelo grupo
    # 0.01 = pequeno, 0.06 = médio, 0.14 = grande
    eta2 = (stat_kw - len(series) + 1) / (n_total - len(series))

    print(f"\nMétodo: {metodo['titulo']}")
    print(f"  H = {stat_kw:.4f}  |  p = {p_kw:.6f}  |  η² = {eta2:.4f}")
    if eta2 < 0.01:
        print("  → Efeito negligenciável — unificação justificada mesmo com p < 0.05")
    elif eta2 < 0.06:
        print("  → Efeito pequeno — unificação provavelmente aceitável")
    elif eta2 < 0.14:
        print("  → Efeito médio — avaliar com cautela")
    else:
        print("  → Efeito grande — distribuições substancialmente diferentes")

### Teste de normalidade Q-Q Plot

In [ ]:
# df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

serie = df["Resultado de Ferro (ppm)"].dropna()

# Visualização
fig_normalidade_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie.min(), serie.max(), 300)
y_normal = stats.norm.pdf(x_range, serie.mean(), serie.std())
y_normal_scaled = y_normal * len(serie) * (serie.max() - serie.min()) / 50

fig_normalidade_crystallizer1.add_trace(go.Histogram(
    x=serie, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_normalidade_crystallizer1.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_normalidade_crystallizer1.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_normalidade_crystallizer1.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_normalidade_crystallizer1.update_yaxes(title_text="Contagem", row=1, col=2)

fig_normalidade_crystallizer1.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade — Resultado de Ferro (ppm) Cristallyzer #1"
)
fig_normalidade_crystallizer1.show()

# Unificando bases de dados

## Unificando dados

In [ ]:
# Original
df_crystallizer123 = pd.concat([
    df_crystallizer1.assign(Crystallizer='C1'),
    df_crystallizer2.assign(Crystallizer='C2'),
    df_crystallizer3.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Originais: {df_crystallizer123.shape}")

# Intervalo 0-10
df_crystallizer123_0a10 = pd.concat([
    df_crystallizer1_0a10.assign(Crystallizer='C1'),
    df_crystallizer2_0a10.assign(Crystallizer='C2'),
    df_crystallizer3_0a10.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Intervalo 0-10: {df_crystallizer123_0a10.shape}")

# IQR
df_crystallizer123_iqr = pd.concat([
    df_crystallizer1_iqr.assign(Crystallizer='C1'),
    df_crystallizer2_iqr.assign(Crystallizer='C2'),
    df_crystallizer3_iqr.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"IQR: {df_crystallizer123_iqr.shape}")

# Hampel
df_crystallizer123_hampel = pd.concat([
    df_crystallizer1_hampel.assign(Crystallizer='C1'),
    df_crystallizer2_hampel.assign(Crystallizer='C2'),
    df_crystallizer3_hampel.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Hampel: {df_crystallizer123_hampel.shape}")

## Unificando eventos

In [ ]:
# Eventos unificados — igual para todos os tratamentos
df_eventos_crystallizer123 = pd.concat([
    df_eventos_crystallizer1.assign(Crystallizer='C1'),
    df_eventos_crystallizer2.assign(Crystallizer='C2'),
    df_eventos_crystallizer3.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')

INTERVALO_MIN_DIAS = 15

df_real1 = df_eventos_crystallizer123[df_eventos_crystallizer123['Real'] == 1].copy()
df_real0 = df_eventos_crystallizer123[df_eventos_crystallizer123['Real'] == 0].sort_values('TIMESTAMP').copy()

real0_filtrados = []
ultimo_ts       = None

for _, row in df_real0.iterrows():
    ts_atual = row['TIMESTAMP']

    if ultimo_ts is None or (ts_atual - ultimo_ts).days > INTERVALO_MIN_DIAS:
        real0_filtrados.append(row)
        ultimo_ts = ts_atual
    # se intervalo <= 10 dias, descarta e mantém o que veio primeiro

df_real0_limpo = pd.DataFrame(real0_filtrados)

print(f"Real=0 antes : {len(df_real0)}")
print(f"Real=0 depois: {len(df_real0_limpo)}")
print(f"Removidos    : {len(df_real0) - len(df_real0_limpo)}")

df_eventos_crystallizer123 = pd.concat([df_real1, df_real0_limpo], ignore_index=True).sort_values('TIMESTAMP').reset_index(drop=True)

print(f"\nDistribuição final:")
print(df_eventos_crystallizer123['Real'].value_counts().to_string())
print(f"Total de eventos: {len(df_eventos_crystallizer123)}")

## Violin Plot das classes

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer123},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer123_0a10},
    {"titulo": "IQR",              "df": df_crystallizer123_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer123_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer123.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1#2#3"
)
fig.show()

# Estatísticas descritivas dos eventos

## Crystallizer #1

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer1_0a10},
    {"titulo": "IQR",              "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer1_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer1.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

resultados = []
for DIAS_JANELA in JANELAS:
    vals = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer1.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) >= 2:
            vals[int(evento["Real"])].extend(v.tolist())

    v0, v1 = np.array(vals[0]), np.array(vals[1])

    # Mann-Whitney: diferença de medianas
    stat_mw, p_mw = mannwhitneyu(v0, v1, alternative='two-sided')
    # Kolmogorov-Smirnov: diferença na forma inteira da distribuição
    stat_ks, p_ks = ks_2samp(v0, v1)
    # Effect size (rank-biserial)
    effect = 1 - (2 * stat_mw) / (len(v0) * len(v1))

    resultados.append({
        'janela':      f"{DIAS_JANELA}D",
        'p_mw':        round(p_mw, 4),
        'p_ks':        round(p_ks, 4),
        'effect_size': round(abs(effect), 4),
        'media_r0':    round(np.mean(v0), 3),
        'media_r1':    round(np.mean(v1), 3),
        'delta_media': round(np.mean(v1) - np.mean(v0), 3),
    })

df_testes = pd.DataFrame(resultados)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

STATS_FUNCS = {
    'media':    np.mean,
    'mediana':  np.median,
    'std':      np.std,
    'max':      np.max,
    'p75':      lambda x: np.percentile(x, 75),
    'p90':      lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range':    lambda x: np.max(x) - np.min(x),
}

registros = []

for DIAS_JANELA in JANELAS:
    stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}

    for _, evento in df_eventos_crystallizer1.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) < 2:
            continue
        classe = int(evento["Real"])
        for nome_stat, func in STATS_FUNCS.items():
            stat_vals[nome_stat][classe].append(func(v))

    for nome_stat in STATS_FUNCS:
        v0 = np.array(stat_vals[nome_stat][0])
        v1 = np.array(stat_vals[nome_stat][1])
        if len(v0) < 2 or len(v1) < 2:
            continue
        stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
        effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
        registros.append({
            'janela':      f"{DIAS_JANELA}d",
            'feature':     nome_stat,
            'effect_size': round(effect, 4),
            'p_value':     round(p, 4),
        })

df_effect = pd.DataFrame(registros)

# Heatmap de effect size
pivot = df_effect.pivot(index='feature', columns='janela', values='effect_size')
pivot = pivot[[f"{d}d" for d in JANELAS]]  # ordena colunas

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    text=np.round(pivot.values, 3),
    texttemplate="%{text}",
    colorbar=dict(title="Effect Size")
))
fig.update_layout(
    title="Effect Size (Mann-Whitney) por Feature e Janela<br>"
          "<sup>Verde = alta separabilidade entre Real=0 e Real=1</sup>",
    template="plotly_white",
    xaxis_title="Janela", yaxis_title="Feature"
)
fig.show()

### Análise por BoxPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}
n_metodos = len(metodos)

fig = make_subplots(
    rows=n_metodos, cols=len(JANELAS),
    subplot_titles=[
        f"{m['titulo']} — {d} dias"
        for m in metodos
        for d in JANELAS
    ],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for col_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
        dados_por_classe = {0: [], 1: []}

        for _, evento in df_eventos_crystallizer1.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
            valores = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(valores) < 2:
                continue
            classe = int(evento["Real"])
            dados_por_classe[classe].extend(valores.tolist())

        for classe, valores in dados_por_classe.items():
            fig.add_trace(go.Box(
                y=valores,
                name=nomes_classe[classe],
                marker_color=cores_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and col_idx == 1),
                boxmean='sd'
            ), row=row_idx, col=col_idx)

        fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
        fig.update_xaxes(title_text=metodo["titulo"], row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    boxmode="group",
    title="Distribuição Fe (ppm): Real vs Falso Positivo por Janela — Crystallizer #1"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer1_filtrado_until2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 (Eventos até 01/04/2020)"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer1_filtrado_after2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 (Eventos após 01/04/2020)"
)
fig.show()

## Crystallizer #2

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer2_0a10},
    {"titulo": "IQR",              "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer2_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer2.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

resultados = []
for DIAS_JANELA in JANELAS:
    vals = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer2.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) >= 2:
            vals[int(evento["Real"])].extend(v.tolist())

    v0, v1 = np.array(vals[0]), np.array(vals[1])

    # Mann-Whitney: diferença de medianas
    stat_mw, p_mw = mannwhitneyu(v0, v1, alternative='two-sided')
    # Kolmogorov-Smirnov: diferença na forma inteira da distribuição
    stat_ks, p_ks = ks_2samp(v0, v1)
    # Effect size (rank-biserial)
    effect = 1 - (2 * stat_mw) / (len(v0) * len(v1))

    resultados.append({
        'janela':      f"{DIAS_JANELA}D",
        'p_mw':        round(p_mw, 4),
        'p_ks':        round(p_ks, 4),
        'effect_size': round(abs(effect), 4),
        'media_r0':    round(np.mean(v0), 3),
        'media_r1':    round(np.mean(v1), 3),
        'delta_media': round(np.mean(v1) - np.mean(v0), 3),
    })

df_testes = pd.DataFrame(resultados)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

STATS_FUNCS = {
    'media':    np.mean,
    'mediana':  np.median,
    'std':      np.std,
    'max':      np.max,
    'p75':      lambda x: np.percentile(x, 75),
    'p90':      lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range':    lambda x: np.max(x) - np.min(x),
}

registros = []

for DIAS_JANELA in JANELAS:
    stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}

    for _, evento in df_eventos_crystallizer2.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) < 2:
            continue
        classe = int(evento["Real"])
        for nome_stat, func in STATS_FUNCS.items():
            stat_vals[nome_stat][classe].append(func(v))

    for nome_stat in STATS_FUNCS:
        v0 = np.array(stat_vals[nome_stat][0])
        v1 = np.array(stat_vals[nome_stat][1])
        if len(v0) < 2 or len(v1) < 2:
            continue
        stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
        effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
        registros.append({
            'janela':      f"{DIAS_JANELA}d",
            'feature':     nome_stat,
            'effect_size': round(effect, 4),
            'p_value':     round(p, 4),
        })

df_effect = pd.DataFrame(registros)

# Heatmap de effect size
pivot = df_effect.pivot(index='feature', columns='janela', values='effect_size')
pivot = pivot[[f"{d}d" for d in JANELAS]]  # ordena colunas

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    text=np.round(pivot.values, 3),
    texttemplate="%{text}",
    colorbar=dict(title="Effect Size")
))
fig.update_layout(
    title="Effect Size (Mann-Whitney) por Feature e Janela<br>"
          "<sup>Verde = alta separabilidade entre Real=0 e Real=1</sup>",
    template="plotly_white",
    xaxis_title="Janela", yaxis_title="Feature"
)
fig.show()

### Análise por BoxPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}
n_metodos = len(metodos)

fig = make_subplots(
    rows=n_metodos, cols=len(JANELAS),
    subplot_titles=[
        f"{m['titulo']} — {d} dias"
        for m in metodos
        for d in JANELAS
    ],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for col_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
        dados_por_classe = {0: [], 1: []}

        for _, evento in df_eventos_crystallizer2.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
            valores = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(valores) < 2:
                continue
            classe = int(evento["Real"])
            dados_por_classe[classe].extend(valores.tolist())

        for classe, valores in dados_por_classe.items():
            fig.add_trace(go.Box(
                y=valores,
                name=nomes_classe[classe],
                marker_color=cores_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and col_idx == 1),
                boxmean='sd'
            ), row=row_idx, col=col_idx)

        fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
        fig.update_xaxes(title_text=metodo["titulo"], row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    boxmode="group",
    title="Distribuição Fe (ppm): Real vs Falso Positivo por Janela — Crystallizer #2"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer2_filtrado_until2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 (Eventos até 01/04/2020)"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer2_filtrado_after2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 (Eventos após 01/04/2020)"
)
fig.show()

## Crystallizer #3

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer3_0a10},
    {"titulo": "IQR",              "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer3_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer3.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

resultados = []
for DIAS_JANELA in JANELAS:
    vals = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer3.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) >= 2:
            vals[int(evento["Real"])].extend(v.tolist())

    v0, v1 = np.array(vals[0]), np.array(vals[1])

    # Mann-Whitney: diferença de medianas
    stat_mw, p_mw = mannwhitneyu(v0, v1, alternative='two-sided')
    # Kolmogorov-Smirnov: diferença na forma inteira da distribuição
    stat_ks, p_ks = ks_2samp(v0, v1)
    # Effect size (rank-biserial)
    effect = 1 - (2 * stat_mw) / (len(v0) * len(v1))

    resultados.append({
        'janela':      f"{DIAS_JANELA}D",
        'p_mw':        round(p_mw, 4),
        'p_ks':        round(p_ks, 4),
        'effect_size': round(abs(effect), 4),
        'media_r0':    round(np.mean(v0), 3),
        'media_r1':    round(np.mean(v1), 3),
        'delta_media': round(np.mean(v1) - np.mean(v0), 3),
    })

df_testes = pd.DataFrame(resultados)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

STATS_FUNCS = {
    'media':    np.mean,
    'mediana':  np.median,
    'std':      np.std,
    'max':      np.max,
    'p75':      lambda x: np.percentile(x, 75),
    'p90':      lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range':    lambda x: np.max(x) - np.min(x),
}

registros = []

for DIAS_JANELA in JANELAS:
    stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}

    for _, evento in df_eventos_crystallizer3.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) < 2:
            continue
        classe = int(evento["Real"])
        for nome_stat, func in STATS_FUNCS.items():
            stat_vals[nome_stat][classe].append(func(v))

    for nome_stat in STATS_FUNCS:
        v0 = np.array(stat_vals[nome_stat][0])
        v1 = np.array(stat_vals[nome_stat][1])
        if len(v0) < 2 or len(v1) < 2:
            continue
        stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
        effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
        registros.append({
            'janela':      f"{DIAS_JANELA}d",
            'feature':     nome_stat,
            'effect_size': round(effect, 4),
            'p_value':     round(p, 4),
        })

df_effect = pd.DataFrame(registros)

# Heatmap de effect size
pivot = df_effect.pivot(index='feature', columns='janela', values='effect_size')
pivot = pivot[[f"{d}d" for d in JANELAS]]  # ordena colunas

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    text=np.round(pivot.values, 3),
    texttemplate="%{text}",
    colorbar=dict(title="Effect Size")
))
fig.update_layout(
    title="Effect Size (Mann-Whitney) por Feature e Janela<br>"
          "<sup>Verde = alta separabilidade entre Real=0 e Real=1</sup>",
    template="plotly_white",
    xaxis_title="Janela", yaxis_title="Feature"
)
fig.show()

### Análise por BoxPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}
n_metodos = len(metodos)

fig = make_subplots(
    rows=n_metodos, cols=len(JANELAS),
    subplot_titles=[
        f"{m['titulo']} — {d} dias"
        for m in metodos
        for d in JANELAS
    ],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for col_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
        dados_por_classe = {0: [], 1: []}

        for _, evento in df_eventos_crystallizer3.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
            valores = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(valores) < 2:
                continue
            classe = int(evento["Real"])
            dados_por_classe[classe].extend(valores.tolist())

        for classe, valores in dados_por_classe.items():
            fig.add_trace(go.Box(
                y=valores,
                name=nomes_classe[classe],
                marker_color=cores_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and col_idx == 1),
                boxmean='sd'
            ), row=row_idx, col=col_idx)

        fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
        fig.update_xaxes(title_text=metodo["titulo"], row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    boxmode="group",
    title="Distribuição Fe (ppm): Real vs Falso Positivo por Janela — Crystallizer #3"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer3_filtrado_until2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 (Eventos até 01/04/2020)"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer3_filtrado_after2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 (Eventos após 01/04/2020)"
)
fig.show()

# Comparação dos 3 reatores

## Verificando effect size por reator

In [ ]:
crystallizers_config = [
    (df_crystallizer1,        df_eventos_crystallizer1, "C1 Original"),
    (df_crystallizer1_0a10,   df_eventos_crystallizer1, "C1 0-10"),
    (df_crystallizer1_iqr,    df_eventos_crystallizer1, "C1 IQR"),
    (df_crystallizer1_hampel, df_eventos_crystallizer1, "C1 Hampel"),
    (df_crystallizer2,        df_eventos_crystallizer2, "C2 Original"),
    (df_crystallizer2_0a10,   df_eventos_crystallizer2, "C2 0-10"),
    (df_crystallizer2_iqr,    df_eventos_crystallizer2, "C2 IQR"),
    (df_crystallizer2_hampel, df_eventos_crystallizer2, "C2 Hampel"),
    (df_crystallizer3,        df_eventos_crystallizer3, "C3 Original"),
    (df_crystallizer3_0a10,   df_eventos_crystallizer3, "C3 0-10"),
    (df_crystallizer3_iqr,    df_eventos_crystallizer3, "C3 IQR"),
    (df_crystallizer3_hampel, df_eventos_crystallizer3, "C3 Hampel"),
]

STATS_FUNCS = {
    'media'   : np.mean,
    'mediana' : np.median,
    'std'     : np.std,
    'max'     : np.max,
    'p75'     : lambda x: np.percentile(x, 75),
    'p90'     : lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range'   : lambda x: np.max(x) - np.min(x),
}

registros_todos = []
for df_c, df_ev, nome_c in crystallizers_config:
    for DIAS_JANELA in JANELAS:
        stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}
        for _, evento in df_ev.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_c['TIMESTAMP'] >= inicio) & (df_c['TIMESTAMP'] < ts)
            v      = df_c[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) < 2:
                continue
            classe = int(evento["Real"])
            for nome_stat, func in STATS_FUNCS.items():
                stat_vals[nome_stat][classe].append(func(v))

        for nome_stat in STATS_FUNCS:
            v0 = np.array(stat_vals[nome_stat][0])
            v1 = np.array(stat_vals[nome_stat][1])
            if len(v0) < 2 or len(v1) < 2:
                continue
            stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
            effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
            registros_todos.append({
                'crystallizer': nome_c,
                'janela'      : f"{DIAS_JANELA}d",
                'feature'     : nome_stat,
                'effect_size' : round(effect, 4),
                'p_value'     : round(p, 4),
            })

df_effect_todos = pd.DataFrame(registros_todos)

# Heatmap — 3 crystallizers × 4 métodos = 12 subplots
n_cols = 4  # Original, 0-10, IQR, Hampel
n_rows = 3  # C1, C2, C3
nomes  = [c[2] for c in crystallizers_config]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=nomes,
    vertical_spacing=0.08
)

for idx, (_, _, nome_c) in enumerate(crystallizers_config):
    row_idx = idx // n_cols + 1
    col_idx = idx % n_cols + 1

    subset = df_effect_todos[df_effect_todos['crystallizer'] == nome_c]
    pivot  = subset.pivot(index='feature', columns='janela', values='effect_size')
    pivot  = pivot[[f"{d}d" for d in JANELAS]]

    fig.add_trace(go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='RdYlGn',
        zmin=0, zmax=1,
        text=np.round(pivot.values, 3),
        texttemplate="%{text}",
        showscale=(col_idx == n_cols and row_idx == n_rows)
    ), row=row_idx, col=col_idx)

fig.update_layout(
    title="Effect Size comparativo — C1, C2, C3 × Original, 0-10, IQR, Hampel",
    template="plotly_white",
    height=400 * n_rows
)
fig.show()

## Correlação cruzada entre os crystallizers

In [ ]:
def add_scatter_regressao(fig, x, y, row, col):
    lr = LinearRegression().fit(x.reshape(-1, 1), y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = lr.predict(x_line.reshape(-1, 1))
    r2 = lr.score(x.reshape(-1, 1), y)
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers',
        marker=dict(size=6, opacity=0.6, color='steelblue'),
        showlegend=False
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode='lines',
        line=dict(color='red', width=2),
        showlegend=False
    ), row=row, col=col)
    fig.add_annotation(
        x=x.min(), y=y.max(),
        text=f"R² = {r2:.3f}",
        showarrow=False,
        font=dict(size=12, color='red'),
        xanchor='left',
        row=row, col=col
    )

freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

    print(f"\n{'='*55}")
    print(f"Método: {m['titulo']}")
    print(m["df_corr"].corr(method="pearson").round(3).to_string())


n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=3,
    subplot_titles=[
        f"{a} vs {b} — {m['titulo']}"
        for m in metodos
        for a, b in pares
    ],
    vertical_spacing=0.06
)

for row_idx, m in enumerate(metodos, start=1):
    for col_idx, (a, b) in enumerate(pares, start=1):
        add_scatter_regressao(fig, m["df_corr"][a].values, m["df_corr"][b].values, row=row_idx, col=col_idx)
        fig.update_xaxes(title_text=a, row=row_idx, col=col_idx)
        fig.update_yaxes(title_text=b, row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template='plotly_white',
    title="Correlação cruzada — C1, C2, C3 × Original, 0-10, IQR, Hampel"
)
fig.show()

## Correlação cruzada com lags
Verificando se um reator tem influência sobre outro

In [ ]:
MAX_LAG = 15
lags = range(-MAX_LAG, MAX_LAG + 1)
freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

# Prepara df_corr para cada método
for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

# Cross-correlação com lag
n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Cross-correlação com Defasagem — {m['titulo']}" for m in metodos],
    vertical_spacing=0.08
)

for row_idx, m in enumerate(metodos, start=1):
    df_c = m["df_corr"]
    resultados_lag = []

    for a, b in pares:
        x = df_c[a].values
        y = df_c[b].values
        for lag in lags:
            if lag < 0:
                xs, ys = x[:lag],  y[-lag:]
            elif lag > 0:
                xs, ys = x[lag:],  y[:-lag]
            else:
                xs, ys = x, y
            r, p = pearsonr(xs, ys)
            resultados_lag.append({
                "par": f"{a} vs {b}", "lag_dias": lag * 3,
                "r": round(r, 4), "p": round(p, 6)
            })

    df_lag = pd.DataFrame(resultados_lag)

    for par in df_lag["par"].unique():
        sub = df_lag[df_lag["par"] == par]
        fig.add_trace(go.Scatter(
            x=sub["lag_dias"], y=sub["r"],
            mode="lines", name=par,
            line=dict(width=2),
            legendgroup=par,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.add_hline(y=0, line_color="gray", line_width=0.5, row=row_idx, col=1)
    fig.update_yaxes(title_text="Pearson r", row=row_idx, col=1)
    fig.update_xaxes(title_text="Defasagem (dias)", row=row_idx, col=1)

    # Lag de máxima correlação
    print(f"\nMétodo: {m['titulo']} — Lag de máxima correlação:")
    for par in df_lag["par"].unique():
        sub  = df_lag[df_lag["par"] == par]
        best = sub.loc[sub["r"].idxmax()]
        print(f"  {par}: lag={best['lag_dias']:.0f} dias  |  r={best['r']:.4f}")

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    hovermode="x unified",
    title="Correlação cruzada com Defasagem entre Reatores<br>"
          "<sup>Pico em lag≠0 indica que um reator influencia o outro, negativo: A influencia B  |  positivo: B influencia A</sup>"
)
fig.show()

# Clusterização (Não supervisionada)

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

configs_crystallizer = {
    "C1": {
        "Original":      (df_crystallizer1,        df_eventos_crystallizer1),
        "Intervalo 0-10":(df_crystallizer1_0a10,   df_eventos_crystallizer1),
        "IQR":           (df_crystallizer1_iqr,    df_eventos_crystallizer1),
        "Hampel":        (df_crystallizer1_hampel, df_eventos_crystallizer1),
    },
    "C2": {
        "Original":      (df_crystallizer2,        df_eventos_crystallizer2),
        "Intervalo 0-10":(df_crystallizer2_0a10,   df_eventos_crystallizer2),
        "IQR":           (df_crystallizer2_iqr,    df_eventos_crystallizer2),
        "Hampel":        (df_crystallizer2_hampel, df_eventos_crystallizer2),
    },
    "C3": {
        "Original":      (df_crystallizer3,        df_eventos_crystallizer3),
        "Intervalo 0-10":(df_crystallizer3_0a10,   df_eventos_crystallizer3),
        "IQR":           (df_crystallizer3_iqr,    df_eventos_crystallizer3),
        "Hampel":        (df_crystallizer3_hampel, df_eventos_crystallizer3),
    },
    "C123": {
        "Original":      (df_crystallizer123,        df_eventos_crystallizer123),
        "Intervalo 0-10":(df_crystallizer123_0a10,   df_eventos_crystallizer123),
        "IQR":           (df_crystallizer123_iqr,    df_eventos_crystallizer123),
        "Hampel":        (df_crystallizer123_hampel, df_eventos_crystallizer123),
    },
}

def extrair_features(nome_c, df_medicoes, df_eventos, janelas, stats_funcs):
    dataset_linhas = []
    for _, evento in df_eventos.iterrows():
        ts     = evento["TIMESTAMP"]
        classe = int(evento["Real"])
        features = {
            'Crystallizer':     nome_c,
            'TIMESTAMP_Evento': ts,
            'Real':             classe,
        }
        for dias in janelas:
            inicio = ts - pd.Timedelta(days=dias)
            mask   = (df_medicoes['TIMESTAMP'] >= inicio) & \
                     (df_medicoes['TIMESTAMP'] <  ts)
            y_ppm  = df_medicoes[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(y_ppm) < 2:
                for nome_stat in stats_funcs:
                    features[f"{nome_stat}_{dias}d"] = np.nan
                continue
            for nome_stat, func in stats_funcs.items():
                features[f"{nome_stat}_{dias}d"] = float(func(y_ppm))

        if not any(not np.isnan(features.get(f"media_{d}d", np.nan))
                   for d in janelas):
            continue
        dataset_linhas.append(features)

    colunas_meta = ['Crystallizer', 'TIMESTAMP_Evento', 'Real']
    df = pd.DataFrame(dataset_linhas)
    colunas_features = sorted(
        [c for c in df.columns if c not in colunas_meta],
        key=lambda c: (c.split('_')[0], int(c.split('_')[-1].replace('d', '')))
    )
    return df[colunas_meta + colunas_features], colunas_features


def pipeline_clustering(df_feat, colunas_features, label_dataset):
    # Imputação
    imputer  = SimpleImputer(strategy='median')
    X_imp    = pd.DataFrame(
        imputer.fit_transform(df_feat[colunas_features]),
        columns=colunas_features, index=df_feat.index
    )

    # Scaling
    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    # Grid search
    resultados = []
    for n_clusters in range(2, 8):
        modelos = {
            'Hierarchical (ward)':     AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
            'Hierarchical (complete)': AgglomerativeClustering(n_clusters=n_clusters, linkage='complete'),
            'Hierarchical (average)':  AgglomerativeClustering(n_clusters=n_clusters, linkage='average'),
            'GMM':                     GaussianMixture(n_components=n_clusters, random_state=42, n_init=10),
            'Bisecting KMeans':        BisectingKMeans(n_clusters=n_clusters, random_state=42, n_init=10),
            'Spectral':                SpectralClustering(n_clusters=n_clusters,
                                                          affinity='nearest_neighbors',
                                                          n_neighbors=10, random_state=42),
        }
        for nome, modelo in modelos.items():
            labels = modelo.fit_predict(X_scaled)
            ari    = adjusted_rand_score(df_feat['Real'], labels)
            nmi    = normalized_mutual_info_score(df_feat['Real'], labels)
            resultados.append({
                'Dataset': label_dataset, 'Modelo': nome,
                'n_clusters': n_clusters,
                'ARI': round(ari, 4), 'NMI': round(nmi, 4),
            })

    df_grid  = pd.DataFrame(resultados).sort_values('ARI', ascending=False)
    melhor   = df_grid.iloc[0]
    melhor_n = int(melhor['n_clusters'])
    melhor_mod = melhor['Modelo']

    # Clusterização final
    modelos_final = {
        'Hierarchical (ward)':     AgglomerativeClustering(n_clusters=melhor_n, linkage='ward'),
        'Hierarchical (complete)': AgglomerativeClustering(n_clusters=melhor_n, linkage='complete'),
        'Hierarchical (average)':  AgglomerativeClustering(n_clusters=melhor_n, linkage='average'),
        'GMM':                     GaussianMixture(n_components=melhor_n, random_state=42, n_init=10),
        'Bisecting KMeans':        BisectingKMeans(n_clusters=melhor_n, random_state=42, n_init=10),
        'Spectral':                SpectralClustering(n_clusters=melhor_n,
                                                      affinity='nearest_neighbors',
                                                      n_neighbors=10, random_state=42),
    }
    df_feat = df_feat.copy()
    df_feat['Cluster'] = modelos_final[melhor_mod].fit_predict(X_scaled)

    ari_f = adjusted_rand_score(df_feat['Real'], df_feat['Cluster'])
    nmi_f = normalized_mutual_info_score(df_feat['Real'], df_feat['Cluster'])

    return {
        'df_features': df_feat,
        'X_scaled':    X_scaled,
        'df_grid':     df_grid,
        'melhor_mod':  melhor_mod,
        'melhor_n':    melhor_n,
        'ari':         ari_f,
        'nmi':         nmi_f,
    }

## Crystallizer #1

In [ ]:
CRYSTALLIZER = "C1"  # "C1" | "C2" | "C3"

resultados_por_dataset = {}

for nome_dataset, (df_med, df_ev) in configs_crystallizer[CRYSTALLIZER].items():

    df_feat, cols_feat = extrair_features(
        CRYSTALLIZER, df_med, df_ev, JANELAS, STATS_FUNCS_CLUSTERING
    )
    res = pipeline_clustering(df_feat, cols_feat, nome_dataset)
    resultados_por_dataset[nome_dataset] = res

    print(f"\n{'='*55}")
    print(f"{CRYSTALLIZER} — {nome_dataset}")
    print(f"  Melhor modelo : {res['melhor_mod']}  (k={res['melhor_n']})")
    print(f"  ARI           : {res['ari']:.4f}")
    print(f"  NMI           : {res['nmi']:.4f}")
    print("\n  Tabela de contingência:")
    print(pd.crosstab(
        res['df_features']['Cluster'], res['df_features']['Real'],
        rownames=['Cluster'], colnames=['Real'],
        margins=True, margins_name='Total'
    ))

In [ ]:
n_datasets = len(resultados_por_dataset)
nomes_datasets = list(resultados_por_dataset.keys())

fig = make_subplots(
    rows=n_datasets, cols=2,
    subplot_titles=[
        titulo
        for nd in nomes_datasets
        for titulo in [
            f"{nd} — Clusters ({resultados_por_dataset[nd]['melhor_mod']}, "
            f"k={resultados_por_dataset[nd]['melhor_n']})",
            f"{nd} — Label Real"
        ]
    ],
    vertical_spacing=0.06
)

cores_cluster = px.colors.qualitative.Plotly
cores_real    = {'0': '#4878CF', '1': '#D65F5F'}
nomes_real    = {'0': 'Falso Positivo', '1': 'Contaminação Real'}
simbolos      = {'0': 'circle', '1': 'diamond'}

for row_idx, nome_dataset in enumerate(nomes_datasets, start=1):
    res      = resultados_por_dataset[nome_dataset]
    df_feat  = res['df_features']
    X_scaled = res['X_scaled']

    pca     = PCA(n_components=2, random_state=42)
    X_pca   = pca.fit_transform(X_scaled)
    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    df_plot = pd.DataFrame({
        'PC1':     X_pca[:, 0],
        'PC2':     X_pca[:, 1],
        'Cluster': df_feat['Cluster'].astype(str),
        'Real':    df_feat['Real'].astype(str),
    })

    # Subplot esquerdo — clusters
    for cluster_id in sorted(df_plot['Cluster'].unique()):
        mask = df_plot['Cluster'] == cluster_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=f'Cluster {cluster_id}',
            marker=dict(
                size=8,
                color=cores_cluster[int(cluster_id) % len(cores_cluster)],
                opacity=0.8, line=dict(width=0.5, color='white')
            ),
            legendgroup=f'cluster_{cluster_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Subplot direito — label real
    for real_id in ['0', '1']:
        mask = df_plot['Real'] == real_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=nomes_real[real_id],
            marker=dict(
                size=8, color=cores_real[real_id],
                symbol=simbolos[real_id], opacity=0.85,
                line=dict(width=0.5, color='white')
            ),
            legendgroup=f'real_{real_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=2)

    for col in [1, 2]:
        fig.update_xaxes(title_text=f'PC1 ({var_pc1:.1f}%)', row=row_idx, col=col)
        fig.update_yaxes(title_text=f'PC2 ({var_pc2:.1f}%)', row=row_idx, col=col)

fig.update_layout(
    title=f'PCA 2D — {CRYSTALLIZER}',
    template='plotly_white',
    height=500 * n_datasets,
    legend=dict(groupclick='toggleitem')
)
fig.show()

## Crystallizer #2

In [ ]:
CRYSTALLIZER = "C2"  # "C1" | "C2" | "C3"

resultados_por_dataset = {}

for nome_dataset, (df_med, df_ev) in configs_crystallizer[CRYSTALLIZER].items():

    df_feat, cols_feat = extrair_features(
        CRYSTALLIZER, df_med, df_ev, JANELAS, STATS_FUNCS_CLUSTERING
    )
    res = pipeline_clustering(df_feat, cols_feat, nome_dataset)
    resultados_por_dataset[nome_dataset] = res

    print(f"\n{'='*55}")
    print(f"{CRYSTALLIZER} — {nome_dataset}")
    print(f"  Melhor modelo : {res['melhor_mod']}  (k={res['melhor_n']})")
    print(f"  ARI           : {res['ari']:.4f}")
    print(f"  NMI           : {res['nmi']:.4f}")
    print("\n  Tabela de contingência:")
    print(pd.crosstab(
        res['df_features']['Cluster'], res['df_features']['Real'],
        rownames=['Cluster'], colnames=['Real'],
        margins=True, margins_name='Total'
    ))

In [ ]:
n_datasets = len(resultados_por_dataset)
nomes_datasets = list(resultados_por_dataset.keys())

fig = make_subplots(
    rows=n_datasets, cols=2,
    subplot_titles=[
        titulo
        for nd in nomes_datasets
        for titulo in [
            f"{nd} — Clusters ({resultados_por_dataset[nd]['melhor_mod']}, "
            f"k={resultados_por_dataset[nd]['melhor_n']})",
            f"{nd} — Label Real"
        ]
    ],
    vertical_spacing=0.06
)

cores_cluster = px.colors.qualitative.Plotly
cores_real    = {'0': '#4878CF', '1': '#D65F5F'}
nomes_real    = {'0': 'Falso Positivo', '1': 'Contaminação Real'}
simbolos      = {'0': 'circle', '1': 'diamond'}

for row_idx, nome_dataset in enumerate(nomes_datasets, start=1):
    res      = resultados_por_dataset[nome_dataset]
    df_feat  = res['df_features']
    X_scaled = res['X_scaled']

    pca     = PCA(n_components=2, random_state=42)
    X_pca   = pca.fit_transform(X_scaled)
    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    df_plot = pd.DataFrame({
        'PC1':     X_pca[:, 0],
        'PC2':     X_pca[:, 1],
        'Cluster': df_feat['Cluster'].astype(str),
        'Real':    df_feat['Real'].astype(str),
    })

    # Subplot esquerdo — clusters
    for cluster_id in sorted(df_plot['Cluster'].unique()):
        mask = df_plot['Cluster'] == cluster_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=f'Cluster {cluster_id}',
            marker=dict(
                size=8,
                color=cores_cluster[int(cluster_id) % len(cores_cluster)],
                opacity=0.8, line=dict(width=0.5, color='white')
            ),
            legendgroup=f'cluster_{cluster_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Subplot direito — label real
    for real_id in ['0', '1']:
        mask = df_plot['Real'] == real_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=nomes_real[real_id],
            marker=dict(
                size=8, color=cores_real[real_id],
                symbol=simbolos[real_id], opacity=0.85,
                line=dict(width=0.5, color='white')
            ),
            legendgroup=f'real_{real_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=2)

    for col in [1, 2]:
        fig.update_xaxes(title_text=f'PC1 ({var_pc1:.1f}%)', row=row_idx, col=col)
        fig.update_yaxes(title_text=f'PC2 ({var_pc2:.1f}%)', row=row_idx, col=col)

fig.update_layout(
    title=f'PCA 2D — {CRYSTALLIZER}',
    template='plotly_white',
    height=500 * n_datasets,
    legend=dict(groupclick='toggleitem')
)
fig.show()

## Crystallizer #3

In [ ]:
CRYSTALLIZER = "C3"  # "C1" | "C2" | "C3"

resultados_por_dataset = {}

for nome_dataset, (df_med, df_ev) in configs_crystallizer[CRYSTALLIZER].items():

    df_feat, cols_feat = extrair_features(
        CRYSTALLIZER, df_med, df_ev, JANELAS, STATS_FUNCS_CLUSTERING
    )
    res = pipeline_clustering(df_feat, cols_feat, nome_dataset)
    resultados_por_dataset[nome_dataset] = res

    print(f"\n{'='*55}")
    print(f"{CRYSTALLIZER} — {nome_dataset}")
    print(f"  Melhor modelo : {res['melhor_mod']}  (k={res['melhor_n']})")
    print(f"  ARI           : {res['ari']:.4f}")
    print(f"  NMI           : {res['nmi']:.4f}")
    print("\n  Tabela de contingência:")
    print(pd.crosstab(
        res['df_features']['Cluster'], res['df_features']['Real'],
        rownames=['Cluster'], colnames=['Real'],
        margins=True, margins_name='Total'
    ))

In [ ]:
n_datasets = len(resultados_por_dataset)
nomes_datasets = list(resultados_por_dataset.keys())

fig = make_subplots(
    rows=n_datasets, cols=2,
    subplot_titles=[
        titulo
        for nd in nomes_datasets
        for titulo in [
            f"{nd} — Clusters ({resultados_por_dataset[nd]['melhor_mod']}, "
            f"k={resultados_por_dataset[nd]['melhor_n']})",
            f"{nd} — Label Real"
        ]
    ],
    vertical_spacing=0.06
)

cores_cluster = px.colors.qualitative.Plotly
cores_real    = {'0': '#4878CF', '1': '#D65F5F'}
nomes_real    = {'0': 'Falso Positivo', '1': 'Contaminação Real'}
simbolos      = {'0': 'circle', '1': 'diamond'}

for row_idx, nome_dataset in enumerate(nomes_datasets, start=1):
    res      = resultados_por_dataset[nome_dataset]
    df_feat  = res['df_features']
    X_scaled = res['X_scaled']

    pca     = PCA(n_components=2, random_state=42)
    X_pca   = pca.fit_transform(X_scaled)
    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    df_plot = pd.DataFrame({
        'PC1':     X_pca[:, 0],
        'PC2':     X_pca[:, 1],
        'Cluster': df_feat['Cluster'].astype(str),
        'Real':    df_feat['Real'].astype(str),
    })

    # Subplot esquerdo — clusters
    for cluster_id in sorted(df_plot['Cluster'].unique()):
        mask = df_plot['Cluster'] == cluster_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=f'Cluster {cluster_id}',
            marker=dict(
                size=8,
                color=cores_cluster[int(cluster_id) % len(cores_cluster)],
                opacity=0.8, line=dict(width=0.5, color='white')
            ),
            legendgroup=f'cluster_{cluster_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Subplot direito — label real
    for real_id in ['0', '1']:
        mask = df_plot['Real'] == real_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=nomes_real[real_id],
            marker=dict(
                size=8, color=cores_real[real_id],
                symbol=simbolos[real_id], opacity=0.85,
                line=dict(width=0.5, color='white')
            ),
            legendgroup=f'real_{real_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=2)

    for col in [1, 2]:
        fig.update_xaxes(title_text=f'PC1 ({var_pc1:.1f}%)', row=row_idx, col=col)
        fig.update_yaxes(title_text=f'PC2 ({var_pc2:.1f}%)', row=row_idx, col=col)

fig.update_layout(
    title=f'PCA 2D — {CRYSTALLIZER}',
    template='plotly_white',
    height=500 * n_datasets,
    legend=dict(groupclick='toggleitem')
)
fig.show()

## Crystallizer #1 #2 #3

In [ ]:
CRYSTALLIZER = "C123"  # "C1" | "C2" | "C3"

resultados_por_dataset = {}

for nome_dataset, (df_med, df_ev) in configs_crystallizer[CRYSTALLIZER].items():

    df_feat, cols_feat = extrair_features(
        CRYSTALLIZER, df_med, df_ev, JANELAS, STATS_FUNCS_CLUSTERING
    )
    res = pipeline_clustering(df_feat, cols_feat, nome_dataset)
    resultados_por_dataset[nome_dataset] = res

    print(f"\n{'='*55}")
    print(f"{CRYSTALLIZER} — {nome_dataset}")
    print(f"  Melhor modelo : {res['melhor_mod']}  (k={res['melhor_n']})")
    print(f"  ARI           : {res['ari']:.4f}")
    print(f"  NMI           : {res['nmi']:.4f}")
    print("\n  Tabela de contingência:")
    print(pd.crosstab(
        res['df_features']['Cluster'], res['df_features']['Real'],
        rownames=['Cluster'], colnames=['Real'],
        margins=True, margins_name='Total'
    ))

In [ ]:
n_datasets = len(resultados_por_dataset)
nomes_datasets = list(resultados_por_dataset.keys())

fig = make_subplots(
    rows=n_datasets, cols=2,
    subplot_titles=[
        titulo
        for nd in nomes_datasets
        for titulo in [
            f"{nd} — Clusters ({resultados_por_dataset[nd]['melhor_mod']}, "
            f"k={resultados_por_dataset[nd]['melhor_n']})",
            f"{nd} — Label Real"
        ]
    ],
    vertical_spacing=0.06
)

cores_cluster = px.colors.qualitative.Plotly
cores_real    = {'0': '#4878CF', '1': '#D65F5F'}
nomes_real    = {'0': 'Falso Positivo', '1': 'Contaminação Real'}
simbolos      = {'0': 'circle', '1': 'diamond'}

for row_idx, nome_dataset in enumerate(nomes_datasets, start=1):
    res      = resultados_por_dataset[nome_dataset]
    df_feat  = res['df_features']
    X_scaled = res['X_scaled']

    pca     = PCA(n_components=2, random_state=42)
    X_pca   = pca.fit_transform(X_scaled)
    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    df_plot = pd.DataFrame({
        'PC1':     X_pca[:, 0],
        'PC2':     X_pca[:, 1],
        'Cluster': df_feat['Cluster'].astype(str),
        'Real':    df_feat['Real'].astype(str),
    })

    # Subplot esquerdo — clusters
    for cluster_id in sorted(df_plot['Cluster'].unique()):
        mask = df_plot['Cluster'] == cluster_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=f'Cluster {cluster_id}',
            marker=dict(
                size=8,
                color=cores_cluster[int(cluster_id) % len(cores_cluster)],
                opacity=0.8, line=dict(width=0.5, color='white')
            ),
            legendgroup=f'cluster_{cluster_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Subplot direito — label real
    for real_id in ['0', '1']:
        mask = df_plot['Real'] == real_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=nomes_real[real_id],
            marker=dict(
                size=8, color=cores_real[real_id],
                symbol=simbolos[real_id], opacity=0.85,
                line=dict(width=0.5, color='white')
            ),
            legendgroup=f'real_{real_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=2)

    for col in [1, 2]:
        fig.update_xaxes(title_text=f'PC1 ({var_pc1:.1f}%)', row=row_idx, col=col)
        fig.update_yaxes(title_text=f'PC2 ({var_pc2:.1f}%)', row=row_idx, col=col)

fig.update_layout(
    title=f'PCA 2D — {CRYSTALLIZER}',
    template='plotly_white',
    height=500 * n_datasets,
    legend=dict(groupclick='toggleitem')
)
fig.show()

# Classificação (Abordagem Supervisionada)

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    make_scorer, f1_score, recall_score, precision_score,
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
    classification_report
)
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')


JANELAS   = [15, 12, 9, 6, 3]
N_SPLITS  = 5       # folds StratifiedKFold
RANDOM_STATE = 42

STATS_FUNCS_CLUSTERING = {
    'media':   np.mean,
    'mediana': np.median,
    'std':     np.std,
    'max':     np.max,
    'p75':     lambda x: np.percentile(x, 75),
    'p90':     lambda x: np.percentile(x, 90),
    'range':   lambda x: np.max(x) - np.min(x),
}

# Mapeamento de datasets por crystallizer e tratamento
CONFIGS = {
    "C1": {
        "Original":       (df_crystallizer1,        df_eventos_crystallizer1),
        "Intervalo 0-10": (df_crystallizer1_0a10,   df_eventos_crystallizer1),
        "IQR":            (df_crystallizer1_iqr,    df_eventos_crystallizer1),
        "Hampel":         (df_crystallizer1_hampel, df_eventos_crystallizer1),
    },
    "C2": {
        "Original":       (df_crystallizer2,        df_eventos_crystallizer2),
        "Intervalo 0-10": (df_crystallizer2_0a10,   df_eventos_crystallizer2),
        "IQR":            (df_crystallizer2_iqr,    df_eventos_crystallizer2),
        "Hampel":         (df_crystallizer2_hampel, df_eventos_crystallizer2),
    },
    "C3": {
        "Original":       (df_crystallizer3,        df_eventos_crystallizer3),
        "Intervalo 0-10": (df_crystallizer3_0a10,   df_eventos_crystallizer3),
        "IQR":            (df_crystallizer3_iqr,    df_eventos_crystallizer3),
        "Hampel":         (df_crystallizer3_hampel, df_eventos_crystallizer3),
    },
    # Base unificada — cada crystallizer usa suas medições individuais
    "Unificado": {
        "Original":       None,   # construído dinamicamente abaixo
        "Intervalo 0-10": None,
        "IQR":            None,
        "Hampel":         None,
    },
}

# Mapeamento de medições por crystallizer para uso na extração unificada
MAP_MEDICOES = {
    "Original":       {"C1": df_crystallizer1,        "C2": df_crystallizer2,        "C3": df_crystallizer3},
    "Intervalo 0-10": {"C1": df_crystallizer1_0a10,   "C2": df_crystallizer2_0a10,   "C3": df_crystallizer3_0a10},
    "IQR":            {"C1": df_crystallizer1_iqr,    "C2": df_crystallizer2_iqr,    "C3": df_crystallizer3_iqr},
    "Hampel":         {"C1": df_crystallizer1_hampel, "C2": df_crystallizer2_hampel, "C3": df_crystallizer3_hampel},
}

# =============================================================================
# FUNÇÕES AUXILIARES
# =============================================================================

def extrair_features(crystallizer, df_medicoes, df_eventos,
                     janelas, stats_funcs, map_medicoes_unif=None):
    """
    Extrai features de janelas temporais para cada evento.
    Se map_medicoes_unif for fornecido, usa as medições do crystallizer
    correto para cada evento (modo unificado).
    """
    dataset_linhas = []

    for _, evento in df_eventos.iterrows():
        ts     = evento["TIMESTAMP"]
        classe = int(evento["Real"])
        cryst  = evento.get("Crystallizer", crystallizer)

        # No modo unificado, busca o df de medições do crystallizer do evento
        df_med = map_medicoes_unif[cryst] if map_medicoes_unif else df_medicoes

        features = {
            'Crystallizer':     cryst,
            'TIMESTAMP_Evento': ts,
            'Real':             classe,
        }

        for dias in janelas:
            inicio = ts - pd.Timedelta(days=dias)
            mask   = (df_med['TIMESTAMP'] >= inicio) & (df_med['TIMESTAMP'] < ts)
            y_ppm  = df_med[mask]["Resultado de Ferro (ppm)"].dropna().values

            if len(y_ppm) < 2:
                for nome_stat in stats_funcs:
                    features[f"{nome_stat}_{dias}d"] = np.nan
                continue

            for nome_stat, func in stats_funcs.items():
                features[f"{nome_stat}_{dias}d"] = float(func(y_ppm))

        if not any(not np.isnan(features.get(f"media_{d}d", np.nan))
                   for d in janelas):
            continue

        dataset_linhas.append(features)

    colunas_meta = ['Crystallizer', 'TIMESTAMP_Evento', 'Real']
    df = pd.DataFrame(dataset_linhas)
    colunas_features = sorted(
        [c for c in df.columns if c not in colunas_meta],
        key=lambda c: (c.split('_')[0], int(c.split('_')[-1].replace('d', '')))
    )
    return df[colunas_meta + colunas_features], colunas_features


def preparar_XY(df_features, colunas_features, incluir_crystallizer=False):
    """Prepara X e y sem aplicar transformações que causam vazamento."""
    X = df_features[colunas_features].copy()
    
    if incluir_crystallizer:
        le = LabelEncoder()
        X['Crystallizer_enc'] = le.fit_transform(df_features['Crystallizer'])
        
    y = df_features['Real']
    return X, y


def definir_modelos(y):
    """Define os modelos com os parâmetros adequados para o desbalanceamento."""
    ratio = (y == 0).sum() / max((y == 1).sum(), 1)
    return {
        'Logistic Regression': LogisticRegression(
            class_weight='balanced', C=0.1,
            max_iter=1000, random_state=RANDOM_STATE
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=500, max_depth=5,
            class_weight='balanced', random_state=RANDOM_STATE
        ),
        'XGBoost': XGBClassifier(
            scale_pos_weight=ratio,
            max_depth=3, n_estimators=200,
            learning_rate=0.05, subsample=0.8,
            eval_metric='aucpr', random_state=RANDOM_STATE,
            verbosity=0
        ),
        'SVM RBF': SVC(
            kernel='rbf', class_weight='balanced',
            probability=True, C=1.0, random_state=RANDOM_STATE
        ),
    }


def avaliar_modelos(X, y, modelos, n_splits=N_SPLITS):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        'f1':        make_scorer(f1_score, zero_division=0),
        'pr_auc':    'average_precision',
        'roc_auc':   'roc_auc',
        'recall':    make_scorer(recall_score, zero_division=0),
        'precision': make_scorer(precision_score, zero_division=0),
    }

    resultados = []
    for nome, modelo in modelos.items():
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')), # Imputer AQUI
            ('scaler', StandardScaler()),
            ('clf',    modelo)
        ])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
        resultados.append({
            'Modelo':    nome,
            'F1':        round(scores['test_f1'].mean(), 4),
            'F1_std':    round(scores['test_f1'].std(),  4),
            'PR_AUC':    round(scores['test_pr_auc'].mean(), 4),
            'ROC_AUC':   round(scores['test_roc_auc'].mean(), 4),
            'Recall':    round(scores['test_recall'].mean(), 4),
            'Precision': round(scores['test_precision'].mean(), 4),
        })

    return pd.DataFrame(resultados).sort_values('PR_AUC', ascending=False)


def threshold_tuning(modelo, X, y, n_splits=N_SPLITS):
    """
    Treina o modelo e encontra o threshold ótimo usando previsões out-of-fold (OOF)
    para evitar overfitting na avaliação.
    """
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')), 
        ('scaler', StandardScaler()), 
        ('clf', modelo)
    ])
    
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    # Previsões OOF (sem vazamento)
    probs_oof = cross_val_predict(pipe, X, y, cv=cv, method='predict_proba')[:, 1]

    # Otimiza o threshold nas previsões OOF
    prec, rec, thresholds = precision_recall_curve(y, probs_oof)
    f1_arr = np.where(
        (prec + rec) > 0,
        2 * prec * rec / (prec + rec), 0
    )
    best_idx   = np.argmax(f1_arr)
    best_thr   = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    y_pred_oof = (probs_oof >= best_thr).astype(int)

    # Treina o pipeline final com todos os dados para uso em produção/SHAP
    pipe.fit(X, y)

    return pipe, best_thr, probs_oof, y_pred_oof


# =============================================================================
# EXECUÇÃO PRINCIPAL
# =============================================================================

# Escolha o modo de execução:
#   "C1" | "C2" | "C3"  → crystallizer individual
#   "Unificado"          → todos os crystallizers juntos
MODO = "Unificado"

todos_resultados = []   # acumula métricas de todos os tratamentos
modelos_treinados = {}  # guarda pipeline treinado por tratamento

tratamentos = list(list(CONFIGS.values())[0].keys())

for tratamento in tratamentos:

    print(f"\n{'='*60}")
    print(f"MODO: {MODO}  |  TRATAMENTO: {tratamento}")
    print(f"{'='*60}")

    # ── Extração de features ─────────────────────────────────────────────────
    if MODO == "Unificado":
        df_eventos_unif = df_eventos_crystallizer123  # dataframe unificado de eventos
        df_feat, cols_feat = extrair_features(
            "Unificado", None, df_eventos_unif,
            JANELAS, STATS_FUNCS_CLUSTERING,
            map_medicoes_unif=MAP_MEDICOES[tratamento]
        )
        X, y = preparar_XY(df_feat, cols_feat, incluir_crystallizer=True)
    else:
        df_med, df_ev = CONFIGS[MODO][tratamento]
        df_feat, cols_feat = extrair_features(
            MODO, df_med, df_ev,
            JANELAS, STATS_FUNCS_CLUSTERING
        )
        X, y = preparar_XY(df_feat, cols_feat, incluir_crystallizer=False)

    print(f"  Amostras : {X.shape[0]}  |  Features: {X.shape[1]}")
    print(f"  Real=1   : {(y==1).sum()}  |  Real=0: {(y==0).sum()}")

    # ── Avaliação via CV ─────────────────────────────────────────────────────
    modelos = definir_modelos(y)
    df_cv   = avaliar_modelos(X, y, modelos)
    df_cv.insert(0, 'Tratamento', tratamento)
    df_cv.insert(0, 'Modo',       MODO)
    todos_resultados.append(df_cv)

    print("\n  Resultados CV:")
    print(df_cv[['Modelo','F1','PR_AUC','ROC_AUC','Recall','Precision']]
          .to_string(index=False))

    # ── Melhor modelo — threshold tuning e treino final ──────────────────────
    melhor_nome  = df_cv.iloc[0]['Modelo']
    melhor_modelo = modelos[melhor_nome]

    pipe_final, thr_opt, probs, y_pred_opt = threshold_tuning(melhor_modelo, X, y)

    modelos_treinados[tratamento] = {
        'pipe':      pipe_final,
        'threshold': thr_opt,
        'X':         X,
        'y':         y,
        'probs':     probs,
        'y_pred':    y_pred_opt,
        'df_feat':   df_feat,
        'melhor':    melhor_nome,
    }

    print(f"\n  Melhor modelo : {melhor_nome}")
    print(f"  Threshold ótimo: {thr_opt:.3f}")
    print(f"\n  Relatório (threshold otimizado):")
    print(classification_report(
        y, y_pred_opt,
        target_names=['Falso Positivo', 'Contaminação Real'],
        zero_division=0
    ))

# =============================================================================
# TABELA COMPARATIVA FINAL
# =============================================================================

df_final = pd.concat(todos_resultados, ignore_index=True)

print("\n" + "="*60)
print("COMPARATIVO FINAL — todos os tratamentos")
print("="*60)
print(df_final[['Tratamento','Modelo','F1','PR_AUC','ROC_AUC','Recall','Precision']]
      .sort_values('PR_AUC', ascending=False)
      .to_string(index=False))

# =============================================================================
# VISUALIZAÇÕES
# =============================================================================

# ── 1. Heatmap de PR-AUC por Modelo × Tratamento ─────────────────────────────
pivot_prauc = df_final.pivot_table(
    index='Modelo', columns='Tratamento', values='PR_AUC'
)

fig_heat = go.Figure(go.Heatmap(
    z=pivot_prauc.values,
    x=pivot_prauc.columns.tolist(),
    y=pivot_prauc.index.tolist(),
    colorscale='RdYlGn', zmin=0, zmax=1,
    text=np.round(pivot_prauc.values, 3),
    texttemplate='%{text}',
    colorbar=dict(title='PR-AUC')
))
fig_heat.update_layout(
    title=f'PR-AUC por Modelo × Tratamento — {MODO}',
    template='plotly_white',
    xaxis_title='Tratamento', yaxis_title='Modelo',
    height=400
)
fig_heat.show()

# ── 2. Curvas PR e ROC para cada tratamento — melhor modelo ──────────────────
fig_curvas = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Curva Precision-Recall', 'Curva ROC']
)
cores = px.colors.qualitative.Plotly

for i, (tratamento, res) in enumerate(modelos_treinados.items()):
    y_true = res['y']
    probs  = res['probs']
    cor    = cores[i % len(cores)]

    # PR
    prec, rec, _ = precision_recall_curve(y_true, probs)
    ap = average_precision_score(y_true, probs)
    fig_curvas.add_trace(go.Scatter(
        x=rec, y=prec, mode='lines',
        name=f"{tratamento} (AP={ap:.3f})",
        line=dict(color=cor, width=2),
        legendgroup=tratamento
    ), row=1, col=1)

    # ROC
    fpr, tpr, _ = roc_curve(y_true, probs)
    roc_auc_val = auc(fpr, tpr)
    fig_curvas.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f"{tratamento} (AUC={roc_auc_val:.3f})",
        line=dict(color=cor, width=2),
        legendgroup=tratamento,
        showlegend=False
    ), row=1, col=2)

fig_curvas.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    line=dict(color='gray', dash='dash', width=1),
    showlegend=False
), row=1, col=2)

fig_curvas.update_xaxes(title_text='Recall',              row=1, col=1)
fig_curvas.update_yaxes(title_text='Precision',            row=1, col=1)
fig_curvas.update_xaxes(title_text='Taxa Falso Positivo',  row=1, col=2)
fig_curvas.update_yaxes(title_text='Taxa Verdadeiro Positivo', row=1, col=2)
fig_curvas.update_layout(
    title=f'Curvas PR e ROC — {MODO} (melhor modelo por tratamento)',
    template='plotly_white', height=450
)
fig_curvas.show()

# ── 3. Matrizes de confusão ───────────────────────────────────────────────────
n_trat = len(modelos_treinados)
fig_cm = make_subplots(
    rows=1, cols=n_trat,
    subplot_titles=[
        f"{t}\n{r['melhor']} (thr={r['threshold']:.2f})"
        for t, r in modelos_treinados.items()
    ]
)

for col_idx, (tratamento, res) in enumerate(modelos_treinados.items(), start=1):
    cm = confusion_matrix(res['y'], res['y_pred'])
    fig_cm.add_trace(go.Heatmap(
        z=cm, colorscale='Blues',
        text=cm, texttemplate='%{text}',
        showscale=False,
        x=['Falso +', 'Real'], y=['Falso +', 'Real']
    ), row=1, col=col_idx)
    fig_cm.update_xaxes(title_text='Previsto', row=1, col=col_idx)
    fig_cm.update_yaxes(title_text='Real',     row=1, col=col_idx)

fig_cm.update_layout(
    title=f'Matrizes de Confusão — {MODO} (threshold otimizado)',
    template='plotly_white',
    height=400
)
fig_cm.show()

# ── 4. SHAP — melhor tratamento overall ──────────────────────────────────────
melhor_tratamento = df_final.sort_values('PR_AUC', ascending=False).iloc[0]['Tratamento']
res_shap = modelos_treinados[melhor_tratamento]
clf_shap = res_shap['pipe'].named_steps['clf']
X_shap   = pd.DataFrame(
    res_shap['pipe'].named_steps['scaler'].transform(res_shap['X']),
    columns=res_shap['X'].columns
)

try:
    explainer   = shap.TreeExplainer(clf_shap)
    shap_values = explainer.shap_values(X_shap)
    # Para classificação binária, usa os valores da classe positiva
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    print(f"\nSHAP — melhor tratamento: {melhor_tratamento} ({res_shap['melhor']})")
    shap.summary_plot(sv, X_shap, max_display=15, show=True)
except Exception:
    # Fallback para modelos não-tree (LR, SVM)
    explainer   = shap.LinearExplainer(clf_shap, X_shap)
    shap_values = explainer.shap_values(X_shap)
    shap.summary_plot(shap_values, X_shap, max_display=15, show=True)

## Crystallizer #1

## Crystallizer #2

## Crystallizer #3

## Crystallizer #1 #2 #3